# Solution one : Feature selection + Train + log to MLFlow

# 0.Library

In [ ]:
# general
import pandas as pd
import importlib
import joblib

# ML
from sklearn.model_selection import LeaveOneOut, cross_validate

# MLFlow
import mlflow

# Paths
import sys
sys.path.append('../') 
import constants_data as cd
from features import model_utils as mu
import paths_data as paths

# Reload
importlib.reload(cd)
importlib.reload(mu)
importlib.reload(paths)

/home/smira/myproject/detection_AD_with_VR_data/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'features.model_utils' from '/home/smira/myproject/detection_AD_with_VR_data/src/notebooks/../features/model_utils.py'>

# 1. Paths and Constants

In [ ]:
# Read models path (for saving them)
models_paths_folder = paths.MODELS_DIR_SOL_ONE

# Read dataframe input (train set)
name_input_df = "cleaned_features_13_patients.csv"
df_path = paths.STAGE_THREE_DATA_PREPROCESSING/ name_input_df
df      = pd.read_csv(df_path)
df.head()

,Age,Help_Rating_out_of_5,MoCA_Score,Tutorial_total_reading_time,Tutorial_intensity_reading_time,Tutorial_total_count_hover,Tutorial_total_duration_hover,Tutorial_mean_duration_hover,Tutorial_max_duration_hover,Tutorial_total_count_press,...,Memory_Yaw_std,Memory_Pitch_std,Memory_Roll_std,Memory_Yaw_range,Memory_Roll_range,Memory_dominant_hand_mean_speed,Memory_not_dominant_hand_mean_speed,Memory_dominant_hand_z_range,Memory_not_dominant_hand_y_range,Gender_Male
0,73.0,4,28.0,45.03,0.92,32,20.64,1.03,2.29,8,...,26.23,161.72,177.36,99.05,360.00,0.16,0.03,0.82,0.80,0
1,82.0,1,26.0,338.75,0.99,688,145.79,2.11,20.01,9,...,23.27,161.90,162.61,109.38,359.99,0.06,0.01,0.98,0.75,1
2,62.0,1,27.0,152.90,0.97,61,54.33,1.81,11.62,8,...,30.07,170.64,169.17,248.93,359.98,0.12,0.02,1.05,0.58,1
3,78.0,2,27.0,181.69,0.98,26,91.62,2.78,23.45,9,...,31.96,136.51,170.83,174.64,359.98,0.12,0.02,0.82,0.42,0
4,71.0,2,26.0,190.51,0.98,109,102.68,2.57,21.86,8,...,27.48,169.32,125.66,146.78,359.99,0.11,0.03,0.82,0.64,0


# 3. Data Splitting

In [ ]:
y = df['MoCA_Score'] # train set
df = df.drop(columns=['MoCA_Score'])
X = df # train set

# 4. Fitting and saving Models

In [ ]:
# experiment list
experiments_list = cd.all_experiments_sol_1
loo = LeaveOneOut()

for exp_name, models_list, k_value in experiments_list:

    print(f"================ Experiment: {exp_name} with k={k_value} ================")
    mlflow.set_experiment(f"MoCA_Regression_feaute_selection_{exp_name}_{k_value}")

    for model_name, model_fn in models_list:
        print(f"======== Run: {exp_name}_{model_name} ========")
        
        # create model name to save it later
        model_name = f"{exp_name}_{model_name}_{k_value}"

        with mlflow.start_run(run_name=f"{exp_name}_{model_name}"):

            pipe = mu.make_model_pipeline(
                mu.make_feature_pipeline(exp_name, k_value),
                model_fn()
            )

            scores = cross_validate(
                pipe,
                X,
                y,
                cv=loo,
                scoring={
                    "mae": "neg_mean_absolute_error",
                    "mse": "neg_mean_squared_error",
                    "rmse": "neg_root_mean_squared_error"
                },
                return_train_score=True
            )

            # compute metrics of test set
            mae_test  = -scores["test_mae"].mean()
            mse_test  = -scores["test_mse"].mean()
            rmse_test = -scores["test_rmse"].mean()

            # compute metrics of train set
            mae_train  = -scores["train_mae"].mean()
            mse_train  = -scores["train_mse"].mean()
            rmse_train = -scores["train_rmse"].mean()

            # compute gap (overfitting)
            gap = mse_test - mse_train

            # print
            print(f"Model: {model_name}")
            print(f"MSE of test: {mse_test:.3f}")
            print(f"MSE of train: {mse_train:.3f}")
            print(f"Gap: {gap:.3f}")

            # MLflow logging
            mlflow.log_param("experiment", exp_name)
            mlflow.log_param("model", model_name)

            mlflow.log_metric("mae_test", mae_test)
            mlflow.log_metric("mse_test", mse_test)
            mlflow.log_metric("rmse_test", rmse_test)

            mlflow.log_metric("mae_train", mae_train)
            mlflow.log_metric("mse_train", mse_train)
            mlflow.log_metric("rmse_train", rmse_train)

            mlflow.log_metric("gap", gap)

            # log model
            mlflow.sklearn.log_model(pipe, name="model", skops_trusted_types=["xgboost.sklearn.XGBRegressor", "sklearn.feature_selection._univariate_selection.f_regression"])
            
            # Save model 
            pipe.fit(X, y)
            
            joblib.dump(pipe, models_paths_folder / f"{model_name}.pkl")

            print(f"Model saved : {models_paths_folder / model_name}.pkl")


================ Experiment: all_features with k=0 ================
======== Run: all_features_LinearRegression ========
Model: all_features_LinearRegression_0
MSE of test: 10.711
MSE of train: 0.000
Gap: 10.711
Model saved : ../../models/solution_one/all_features_LinearRegression_0/.pkl
======== Run: all_features_Ridge ========
Model: all_features_Ridge_0
MSE of test: 10.691
MSE of train: 0.000
Gap: 10.691
Model saved : ../../models/solution_one/all_features_Ridge_0/.pkl
======== Run: all_features_RandomForest ========
Model: all_features_RandomForest_0
MSE of test: 10.552
MSE of train: 1.113
Gap: 9.439
Model saved : ../../models/solution_one/all_features_RandomForest_0/.pkl
======== Run: all_features_XGBRegressor ========
Model: all_features_XGBRegressor_0
MSE of test: 25.601
MSE of train: 0.000
Gap: 25.601
Model saved : ../../models/solution_one/all_features_XGBRegressor_0/.pkl
======== Run: all_features_SVR ========
Model: all_features_SVR_0
MSE of test: 11.039
MSE of train: 7.215
